In [1]:
import os
import pandas as pd

# doc cac file csv goc tu thu muc data/raw
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')

# gop cac bang lai
df = pd.merge(orders, customers, on='customer_id', how='inner')
df = pd.merge(df, order_items, on='order_id', how='inner')
df = pd.merge(df, products, on='product_id', how='left')
df = pd.merge(df, sellers, on='seller_id', how='left')
df = pd.merge(df, payments, on='order_id', how='left')
df = pd.merge(df, reviews, on='order_id', how='left')

# chuyen sang kieu datetime de tinh ngay hang
date_cols = [
    'order_purchase_timestamp', 'order_approved_at', 
    'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date'
]
for c in date_cols:
    df[c] = pd.to_datetime(df[c])

# xoa cac dong trung nhau
df = df.drop_duplicates()

# tinh them cac cot total_amount, delivery_days voi order_month
df['total_amount'] = df['price'] + df['freight_value'] 
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_delivered_carrier_date']).dt.days
df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

# loc cac cot dung theo checklist up file final_sales
final_cols = [
    'order_id', 'customer_id', 'customer_unique_id', 'order_purchase_timestamp', 
    'order_month', 'customer_state', 'seller_state', 'product_category_name', 'payment_type', 
    'price', 'freight_value', 'total_amount', 'review_score', 'delivery_days'
]
final_sales = df[final_cols].copy()

# xuat file ra dung thu muc data/processed theo cau truc moi
os.makedirs('../data/processed', exist_ok=True)
final_sales.to_csv('../data/processed/final_sales.csv', index=False)
print("Da xu ly xong va xuat file final_sales.csv ra thu muc data/processed/")

Da xu ly xong va xuat file final_sales.csv ra thu muc data/processed/


In [2]:
import os
import pandas as pd

# doc file tu buoc truoc
df = pd.read_csv('../data/processed/final_sales.csv')
os.makedirs('../data/warehouse', exist_ok=True)

# 1. Tach ra cac bang dimension dung y chang danh sach
dim_customer = df[['customer_id', 'customer_unique_id', 'customer_state']].drop_duplicates()
dim_customer.to_csv('../data/warehouse/dim_customer.csv', index=False)

dim_seller = df[['order_id', 'seller_state']].drop_duplicates()
dim_seller.to_csv('../data/warehouse/dim_seller.csv', index=False)

dim_payment = df[['payment_type']].drop_duplicates().dropna()
dim_payment.to_csv('../data/warehouse/dim_payment.csv', index=False)

dim_product = df[['product_category_name']].drop_duplicates().dropna()
dim_product.to_csv('../data/warehouse/dim_product.csv', index=False)

dim_date = df[['order_id', 'order_month']].drop_duplicates()
dim_date.to_csv('../data/warehouse/dim_date.csv', index=False)

# 2. Tao bang fact
fact_sales = df[['order_id', 'customer_id', 'order_purchase_timestamp', 'price', 'freight_value', 'total_amount', 'review_score', 'delivery_days']].drop_duplicates()
fact_sales.to_csv('../data/warehouse/fact_sales.csv', index=False)

# 3. Join cac bang lai de ra file olap_sales_view.csv cho anh Phuc
required_cols_olap = [
    'order_id', 'customer_unique_id', 'order_month', 'customer_state', 
    'product_category_name', 'payment_type', 'seller_state', 'total_amount', 'review_score', 'delivery_days'
]
olap_sales_view = df[required_cols_olap].drop_duplicates()
olap_sales_view.to_csv('../data/warehouse/olap_sales_view.csv', index=False)

print("Da tao xong cac bang Dim, Fact tai data/warehouse/")

Da tao xong cac bang Dim, Fact tai data/warehouse/
